# LimingjieNN · Colab 训练流水线

| 运行时 | 运行单元 |
| --- | --- |
| **CPU Runtime** | Cell 1 初始化，Cell 2 单测，Cell 3 对账与压测，Cell 4 采样，Cell 7 回放，Cell 8 封存评估 |
| **T4 GPU Runtime** | 重新运行 Cell 1，随后运行 Cell 5 训练、Cell 6 导出 |

在 Cell 1 选择 Drive 的 `repo/` 或私有 GitHub 仓库 `Flysky701/LimingjieNN`。GitHub 模式读取 Colab Secret `LIMINGJIENN_GITHUB_TOKEN`。
切换运行时前确认 Drive 已收到所有 JSONL 分片及 manifest。Round 1 的训练集包含 Round 0 和所有 Round 1 工作者的分片。

In [ ]:
# CPU Runtime / T4 GPU Runtime · repository and shared settings
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/LimingjieNN')
REPO_SOURCE = 'drive'  # 'drive' or 'github'
CN_DB_PATH = DRIVE_ROOT / 'master_cn.db'
CN_DB_SHA256 = ''  # Set to the pinned CN DB digest when releasing a run.
CN_DB_VERSION = '202611131613'

if REPO_SOURCE == 'github':
    from google.colab import userdata
    REPO_DIR = Path('/content/LimingjieNN')
    token = userdata.get('LIMINGJIENN_GITHUB_TOKEN')
    with tempfile.TemporaryDirectory() as auth_dir:
        askpass = Path(auth_dir) / 'git-askpass'
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '  *Username*) printf "%s\\n" x-access-token ;;\n'
            '  *) printf "%s\\n" "$LIMINGJIENN_GITHUB_TOKEN" ;;\n'
            'esac\n', encoding='utf-8',
        )
        askpass.chmod(0o700)
        git_env = dict(os.environ, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0',
                       LIMINGJIENN_GITHUB_TOKEN=token)
        if (REPO_DIR / '.git').exists():
            git_cmd = ['git', '-C', REPO_DIR, 'pull', '--ff-only']
        else:
            git_cmd = ['git', 'clone', '--branch', 'main',
                       'https://github.com/Flysky701/LimingjieNN.git', REPO_DIR]
        subprocess.run(git_cmd, env=git_env, check=True)
    del git_env, token
elif REPO_SOURCE == 'drive':
    REPO_SRC = DRIVE_ROOT / 'repo'
    REPO_DIR = Path('/content/LimingjieNN-drive')
    if not (REPO_SRC / 'pyproject.toml').is_file():
        raise FileNotFoundError(f'Upload the repository to {REPO_SRC}')
    shutil.copytree(REPO_SRC, REPO_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns(
        '.git', '.venv', 'artifacts', '__pycache__', '*.egg-info',
    ))
else:
    raise ValueError('REPO_SOURCE must be drive or github')

CACHE_DIR = DRIVE_ROOT / 'cache'
DATA_DIR = DRIVE_ROOT / 'data'
RUNS_DIR = DRIVE_ROOT / 'runs'
EXPORT_DIR = DRIVE_ROOT / 'export_bundle'
for folder in (CACHE_DIR, DATA_DIR, RUNS_DIR, EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CATALOG_PATH = DATA_DIR / 'feature_pack.json'
ROUND = 0  # Use 1 for Round 1 across both runtimes.
PORT = 8080
SAMPLE_PREFIX = 'round0_pilot' if ROUND == 0 else 'round1_scale'
# For parallel workers set distinct SAMPLE_PREFIX, ROUND_ID, SEED and PORT.
ROUND_ID = SAMPLE_PREFIX
SEED = 20261122 if ROUND == 0 else 20261124
CHUNK_SIZE = 5000 if ROUND == 0 else 2500
TARGET_BATTLES = 2000 if ROUND == 0 else 125000
PROFILE = f"configs/sim_profile_{'highscore' if ROUND == 0 else 'round1'}_cn.json"
SAMPLING = f'configs/sampling_round{ROUND}.json'
TRAIN_PREFIXES = ['round0_pilot'] if ROUND == 0 else ['round0_pilot', SAMPLE_PREFIX]  # List every sampling worker prefix here.
ABLATION_DIR = RUNS_DIR / f'ablation_round{ROUND}'

os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train]'], check=True)

def run_module(module, *args, **options):
    return subprocess.run([sys.executable, '-m', module, *map(str, args)], check=True, **options)

def file_sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

import torch
print(f'CPU count={os.cpu_count()} | CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# CPU Runtime · unit tests, including CPU ONNX Runtime parity
run_module('unittest', 'discover', '-s', 'tests', '-v')

In [ ]:
# CPU Runtime · offline simulator reconciliation and throughput
if not CN_DB_PATH.is_file():
    raise FileNotFoundError(f'Upload the locked CN database to {CN_DB_PATH}')
cmd = ['bash', 'scripts/colab_setup_simulator.sh', '--cache-dir', CACHE_DIR,
       '--db-path', CN_DB_PATH, '--db-version', CN_DB_VERSION, '--port', str(PORT)]
if CN_DB_SHA256:
    cmd.extend(['--db-sha256', CN_DB_SHA256])
subprocess.run(cmd, check=True)
run_module('labyrinth_nn.catalog.build_catalog', '--db', CN_DB_PATH, '--out', CATALOG_PATH)
probe = run_module('labyrinth_nn.data.probe_simulator', '--catalog', CATALOG_PATH,
                   '--profile', PROFILE, '--url', f'http://127.0.0.1:{PORT}',
                   '--bench-count', 64, capture_output=True, text=True)
(DATA_DIR / f'probe_round{ROUND}.json').write_text(probe.stdout, encoding='utf-8')
print(probe.stdout)

In [ ]:
# CPU Runtime · restore all shards, sample locally, and publish completed files
local_dir = Path('/tmp/limingjienn-samples')
local_dir.mkdir(parents=True, exist_ok=True)
manifest_name = f'{SAMPLE_PREFIX}.manifest.json'

def sync_samples(source, destination):
    files = sorted(source.glob(f'{SAMPLE_PREFIX}*.jsonl'))
    if (manifest := source / manifest_name).is_file():
        files.append(manifest)  # Publish after every shard.
    for file in files:
        target = destination / file.name
        if target.is_file():
            if file_sha256(file) == file_sha256(target):
                continue
            if file.suffix == '.jsonl' and not file.name.endswith('.failed.jsonl'):
                raise ValueError(f'Conflicting immutable shard: {target}')
            if target.stat().st_mtime_ns > file.stat().st_mtime_ns:
                continue
        with tempfile.NamedTemporaryFile(dir=destination, prefix=f'.{file.name}.', delete=False) as temporary:
            staged = Path(temporary.name)
        try:
            shutil.copy2(file, staged)
            os.replace(staged, target)
        finally:
            staged.unlink(missing_ok=True)

sampling_args = ('--catalog', CATALOG_PATH, '--profile', PROFILE, '--sampling', SAMPLING,
                 '--seed', SEED, '--round-id', ROUND_ID, '--chunk-size', CHUNK_SIZE)
sync_samples(DATA_DIR, local_dir)
run_module('labyrinth_nn.data.sampler', *sampling_args, '--dry-run', 2000)
try:
    run_module('labyrinth_nn.data.sampler', *sampling_args, '--url', f'http://127.0.0.1:{PORT}',
               '--out', local_dir / f'{SAMPLE_PREFIX}.jsonl', '--target-battles', TARGET_BATTLES, '--resume')
finally:
    sync_samples(local_dir, DATA_DIR)

In [ ]:
# T4 GPU Runtime · train against the complete sorted shard set
import re

def collect_shards(prefix):
    pattern = re.compile(rf'{re.escape(prefix)}(?:[.]part-[0-9]{{5,}})?[.]jsonl')
    files = sorted(p for p in DATA_DIR.glob(f'{prefix}*.jsonl') if pattern.fullmatch(p.name))
    if not files:
        raise FileNotFoundError(f'No shards for {prefix} in {DATA_DIR}')
    manifest = DATA_DIR / f'{prefix}.manifest.json'
    if manifest.is_file():
        entries = json.loads(manifest.read_text(encoding='utf-8'))['shards']
        if {e['filename'] for e in entries} != {p.name for p in files}:
            raise ValueError(f'Shard filenames differ from {manifest}')
        for entry in entries:
            shard = DATA_DIR / entry['filename']
            if file_sha256(shard) != entry['sha256']:
                raise ValueError(f'Shard hash differs from {manifest}: {shard}')
    return files

shards = sorted({shard for prefix in TRAIN_PREFIXES for shard in collect_shards(prefix)})
model_args = ['--configs', 'm0', 'm1', 'm2a', 'm2b']
if ROUND == 1:
    model_args = ['--configs', 'm1', 'm2b', '--config-files',
                  'configs/critic_m2b_r1.json', 'configs/critic_m2b_r1_dmg025.json',
                  '--group-by', 'context']
run_module('labyrinth_nn.train.train_critic', *model_args, '--seeds', 42, '--amp', '--device', 'cuda',
           '--pin-memory', '--catalog', CATALOG_PATH, '--samples', *shards, '--out-dir', ABLATION_DIR)
print((ABLATION_DIR / 'ablation_summary.md').read_text(encoding='utf-8'))

In [ ]:
# T4 GPU Runtime · export and verify the selected checkpoint
WINNING_MODE = 'm2b'  # Set after reviewing the selection report.
WINNING_CKPT = ABLATION_DIR / WINNING_MODE / 'seed_42' / 'best.pt'
BUNDLE_DIR = EXPORT_DIR / WINNING_MODE
run_module('labyrinth_nn.export.export_onnx', '--checkpoint', WINNING_CKPT,
           '--catalog', CATALOG_PATH, '--out-dir', BUNDLE_DIR)

manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('Exported bundle:', BUNDLE_DIR, '| max_abs_diff:', manifest['validation']['max_abs_diff'])

In [ ]:
# CPU Runtime · Optional: Path B global official-score benchmark on recorded opening maps
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

from labyrinth_nn.catalog.schema import FeatureCatalog
from labyrinth_nn.env.run_env import PathBReplayPolicy, compare_policies, load_replays

REPLAY_PATH = DRIVE_ROOT / 'replays.jsonl'  # Optional recorded enter/map_list JSONL
if REPLAY_PATH.exists():
    catalog = FeatureCatalog.load(CATALOG_PATH)
    episodes = load_replays(REPLAY_PATH)
    reports = compare_policies(
        episodes,
        path_b=PathBReplayPolicy(target_score=53000),
        settings=catalog.score_settings,
        catalog=catalog,
        repeats=5,
    )
    for name, rep in reports.items():
        print(name, rep)
else:
    print(f'Place recorded opening maps at {REPLAY_PATH} to run Path B replay comparison.')

In [ ]:
# CPU Runtime · score separately collected sealed samples with a frozen checkpoint
# Set these paths after the panel, model and selection rules have been frozen.
SEALED_SHARDS = []  # All sealed JSONL/Parquet paths, outside TRAIN_PREFIXES.
FROZEN_CHECKPOINT = None

if SEALED_SHARDS and FROZEN_CHECKPOINT:
    from torch.utils.data import DataLoader
    from labyrinth_nn.catalog.schema import FeatureCatalog
    from labyrinth_nn.data.dataset import BattleDataset, collate_battle_batch, mask_gold_sample
    from labyrinth_nn.data.sample_schema import read_samples
    from labyrinth_nn.models.critic import CriticConfig, TwoTowerBattleCritic
    from labyrinth_nn.train.metrics import evaluate_predictions
    from labyrinth_nn.train.train_critic import predict_records

    catalog = FeatureCatalog.load(CATALOG_PATH)
    state = torch.load(FROZEN_CHECKPOINT, map_location='cpu', weights_only=True)
    if state['catalog_sha256'] != (catalog.catalog_sha256 or catalog.compute_sha256()):
        raise ValueError('Sealed panel catalog differs from the frozen checkpoint')
    config = CriticConfig.from_dict(state['config'], catalog)
    model = TwoTowerBattleCritic(config)
    model.load_state_dict(state['model'])
    missing = [Path(shard) for shard in SEALED_SHARDS if not Path(shard).is_file()]
    if missing:
        raise FileNotFoundError(f'Missing sealed panel shards: {missing}')
    samples = [sample for shard in sorted(SEALED_SHARDS) for sample in read_samples(shard)]
    if (not samples or len({s.sample_id for s in samples}) != len(samples)
            or any(s.db_version != catalog.db_version or not sum(s.outcome_counts) for s in samples)):
        raise ValueError('Sealed panel must be nonempty, unique, labeled and on the locked DB version')

    def score(rows):
        dataset = BattleDataset(catalog, rows, roster_max_len=config.roster_max_len,
                                relic_max_len=config.relic_max_len)
        loader = DataLoader(dataset, batch_size=config.batch_size, collate_fn=collate_battle_batch)
        return evaluate_predictions(predict_records(model, loader, 'cpu'))

    report = {'metrics': score(samples),
              'counterfactual_gold_unknown': score([mask_gold_sample(sample) for sample in samples])}
    (RUNS_DIR / 'sealed_test_metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')